# Twilight Imperium Word Document Processor
## Process DOCX files and add to existing Vector Database

This notebook processes Word documents (.docx) and adds them to the existing FAISS vector store.


In [3]:
# Import necessary libraries
from docx import Document
import os
import json
from pathlib import Path
import re
from typing import List, Dict, Any

# LangChain imports for chunking and embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.schema import Document as LangChainDocument

# Load environment variables
from dotenv import load_dotenv
load_dotenv(override=True)

print("✅ All libraries imported successfully")


✅ All libraries imported successfully


In [4]:
# Define paths to our Word document files
dataset_path = Path("dataset")
pravila_docx = dataset_path / "twilight_4_pravila.docx"
faq_docx = dataset_path / "Copy of Official Dane FAQ.docx"

# Verify files exist
print(f"Pravila DOCX exists: {pravila_docx.exists()}")
print(f"FAQ DOCX exists: {faq_docx.exists()}")

# Output directory for processed text
output_dir = Path("processed_rules")
output_dir.mkdir(exist_ok=True)
print(f"Output directory: {output_dir}")


Pravila DOCX exists: True
FAQ DOCX exists: True
Output directory: processed_rules


In [5]:
def extract_text_from_docx(docx_path: Path, source_name: str) -> Dict[str, Any]:
    """
    Extract text from a Word document and return structured data
    
    Args:
        docx_path: Path to the DOCX file
        source_name: Name to identify the source (e.g., 'pravila' or 'faq')
    
    Returns:
        Dictionary containing extracted text and metadata
    """
    print(f"\nProcessing {source_name}...")
    
    # Open the Word document
    doc = Document(docx_path)
    
    # Extract text from paragraphs
    paragraphs = []
    full_text = ""
    
    for i, para in enumerate(doc.paragraphs):
        text = para.text.strip()
        
        if text:  # Only add non-empty paragraphs
            paragraphs.append({
                'paragraph_number': i + 1,
                'text': text
            })
            full_text += text + "\n\n"
    
    # Also extract text from tables if any
    tables_text = []
    for table_idx, table in enumerate(doc.tables):
        table_content = []
        for row in table.rows:
            row_text = " | ".join(cell.text.strip() for cell in row.cells)
            if row_text.strip():
                table_content.append(row_text)
        
        if table_content:
            table_text = "\n".join(table_content)
            tables_text.append(table_text)
            full_text += f"\n\n[Table {table_idx + 1}]\n{table_text}\n\n"
    
    result = {
        'source': source_name,
        'total_paragraphs': len(paragraphs),
        'total_tables': len(tables_text),
        'full_text': full_text.strip(),
        'paragraphs': paragraphs,
        'tables': tables_text
    }
    
    print(f"Extracted text from {len(paragraphs)} paragraphs")
    print(f"Extracted {len(tables_text)} tables")
    print(f"Total characters extracted: {len(full_text)}")
    
    return result


In [6]:
# Extract text from Pravila Word document
pravila_data = extract_text_from_docx(pravila_docx, "twilight_pravila")



Processing twilight_pravila...
Extracted text from 483 paragraphs
Extracted 0 tables
Total characters extracted: 60087


In [7]:
# Extract text from FAQ Word document
faq_data = extract_text_from_docx(faq_docx, "official_faq")



Processing official_faq...
Extracted text from 279 paragraphs
Extracted 0 tables
Total characters extracted: 35744


In [8]:
# Save extracted text to files for record keeping
def save_extracted_data(data: Dict, filename: str):
    """
    Save extracted text data to both JSON and plain text files
    """
    # Save as JSON for structured access
    json_path = output_dir / f"{filename}.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    # Save as plain text for easy reading
    txt_path = output_dir / f"{filename}.txt"
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(data['full_text'])
    
    print(f"Saved {filename} data to:")
    print(f"  JSON: {json_path}")
    print(f"  Text: {txt_path}")

# Save both extracted datasets
save_extracted_data(pravila_data, "twilight_pravila")
save_extracted_data(faq_data, "official_faq")


Saved twilight_pravila data to:
  JSON: processed_rules\twilight_pravila.json
  Text: processed_rules\twilight_pravila.txt
Saved official_faq data to:
  JSON: processed_rules\official_faq.json
  Text: processed_rules\official_faq.txt


In [9]:
# Display summary statistics
print("\n" + "="*50)
print("EXTRACTION SUMMARY")
print("="*50)

print(f"\nTwilight Pravila:")
print(f"  Total paragraphs: {pravila_data['total_paragraphs']}")
print(f"  Total tables: {pravila_data['total_tables']}")
print(f"  Characters extracted: {len(pravila_data['full_text']):,}")

print(f"\nOfficial FAQ:")
print(f"  Total paragraphs: {faq_data['total_paragraphs']}")
print(f"  Total tables: {faq_data['total_tables']}")
print(f"  Characters extracted: {len(faq_data['full_text']):,}")

total_chars = len(pravila_data['full_text']) + len(faq_data['full_text'])
print(f"\nTotal characters from both documents: {total_chars:,}")



EXTRACTION SUMMARY

Twilight Pravila:
  Total paragraphs: 483
  Total tables: 0
  Characters extracted: 60,085

Official FAQ:
  Total paragraphs: 279
  Total tables: 0
  Characters extracted: 35,742

Total characters from both documents: 95,827


In [10]:
# Preview some of the extracted content
print("\n" + "="*50)
print("CONTENT PREVIEW")
print("="*50)

print("\n🔹 Twilight Pravila (first 500 characters):")
print("-" * 40)
print(pravila_data['full_text'][:500] + "...")

print("\n🔹 Official FAQ (first 500 characters):")
print("-" * 40)
print(faq_data['full_text'][:500] + "...")



CONTENT PREVIEW

🔹 Twilight Pravila (first 500 characters):
----------------------------------------
General

Dice Values

Q: Are the “0” faces of the ten-sided dice included with the game intended to represent the result of a “10”?

A: Yes, the “0” face is a “10” result.

Commodities

Q: At the start of the game, do players begin with the commodities listed on their faction sheet?

A: No

Strategy Card - Trade

Q: When the player resolving the “Trade” strategy card selects another player to resolve the “Trade” secondary ability for free, does the player resolving the effect for free do so duri...

🔹 Official FAQ (first 500 characters):
----------------------------------------
General

Q: In maps utilizing hyperlanes (e.g. 5p map), what counts as the edge of the game board?

A: Intention is that one of the edges of a system tile does not touch another system tile OR hyperlane tile in order to be considered on the edge of the board.

Q: When ships move, do they move “into” systems othe

## Step 2: Chunk the Text
Now we'll split the extracted text into chunks for embedding


In [11]:
# Configure the text splitter (same settings as your existing pipeline)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=[
        "\n\n",  # Double newlines (paragraphs)
        "\n",    # Single newlines
        ". ",    # Sentences
        ", ",    # Clauses
        " ",     # Words
        ""       # Characters (last resort)
    ],
    keep_separator=True
)

print("🔧 Text splitter configured:")
print(f"  - Chunk size: {text_splitter._chunk_size}")
print(f"  - Chunk overlap: {text_splitter._chunk_overlap}")


🔧 Text splitter configured:
  - Chunk size: 800
  - Chunk overlap: 100


In [12]:
def create_chunks_with_metadata(text: str, source: str, doc_type: str) -> List[Dict[str, Any]]:
    """
    Split text into chunks and add metadata for better retrieval
    
    Args:
        text: The full text to chunk
        source: Source identifier
        doc_type: Human-readable document type
        
    Returns:
        List of chunks with metadata
    """
    print(f"\n📝 Processing {doc_type}...")
    
    # Split the text into chunks
    chunks = text_splitter.split_text(text)
    
    # Create chunks with metadata
    chunks_with_metadata = []
    
    for i, chunk in enumerate(chunks):
        # Extract potential section headers
        lines = chunk.split('\n')
        potential_header = None
        
        # Look for short lines that might be headers
        for line in lines[:3]:
            if line.strip() and len(line.strip()) < 100:
                if len(line.strip().split()) <= 8:
                    potential_header = line.strip()
                    break
        
        # Create metadata for this chunk
        metadata = {
            'source': source,
            'doc_type': doc_type,
            'chunk_id': f"{source}_chunk_{i:03d}",
            'chunk_index': i,
            'total_chunks': len(chunks),
            'char_count': len(chunk),
            'word_count': len(chunk.split()),
        }
        
        # Add section header if found
        if potential_header:
            metadata['section'] = potential_header
        
        # Store the chunk with its metadata
        chunks_with_metadata.append({
            'content': chunk.strip(),
            'metadata': metadata
        })
    
    print(f"  ✅ Created {len(chunks_with_metadata)} chunks")
    print(f"  📊 Average chunk size: {sum(len(c['content']) for c in chunks_with_metadata) // len(chunks_with_metadata)} characters")
    
    return chunks_with_metadata


In [13]:
# Process Pravila document
pravila_chunks = create_chunks_with_metadata(
    text=pravila_data['full_text'],
    source='twilight_pravila',
    doc_type='Twilight Pravila (Rules)'
)



📝 Processing Twilight Pravila (Rules)...
  ✅ Created 88 chunks
  📊 Average chunk size: 706 characters


In [14]:
# Process FAQ document
faq_chunks = create_chunks_with_metadata(
    text=faq_data['full_text'],
    source='official_faq',
    doc_type='Official FAQ'
)



📝 Processing Official FAQ...
  ✅ Created 53 chunks
  📊 Average chunk size: 692 characters


In [15]:
# Combine all new chunks
all_new_chunks = pravila_chunks + faq_chunks

print("\n" + "="*60)
print("CHUNKING SUMMARY")
print("="*60)

print(f"\n📊 Overall Statistics:")
print(f"  Total new chunks created: {len(all_new_chunks)}")
print(f"  Pravila chunks: {len(pravila_chunks)}")
print(f"  FAQ chunks: {len(faq_chunks)}")

# Calculate size statistics
chunk_sizes = [len(chunk['content']) for chunk in all_new_chunks]
avg_size = sum(chunk_sizes) // len(chunk_sizes) if chunk_sizes else 0
min_size = min(chunk_sizes) if chunk_sizes else 0
max_size = max(chunk_sizes) if chunk_sizes else 0

print(f"\n📏 Chunk Size Distribution:")
print(f"  Average size: {avg_size} characters")
print(f"  Minimum size: {min_size} characters")
print(f"  Maximum size: {max_size} characters")



CHUNKING SUMMARY

📊 Overall Statistics:
  Total new chunks created: 141
  Pravila chunks: 88
  FAQ chunks: 53

📏 Chunk Size Distribution:
  Average size: 701 characters
  Minimum size: 354 characters
  Maximum size: 797 characters


## Step 3: Add to Existing Vector Store
Now we'll load the existing vector store and add the new chunks


In [16]:
# Verify OpenAI API key is set
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    print("❌ OpenAI API key not found!")
    print("Please set your OPENAI_API_KEY environment variable or create a .env file")
elif api_key.startswith("sk-"):
    print("✅ OpenAI API key found and appears valid")
    print(f"Key starts with: {api_key[:20]}...")
else:
    print("⚠️  API key found but format looks incorrect")

# Initialize the embedding model (same as your existing pipeline)
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)
print("✅ OpenAI Embeddings model initialized")


✅ OpenAI API key found and appears valid
Key starts with: sk-proj-gU0tE3P-TebB...
✅ OpenAI Embeddings model initialized


In [17]:
# Load the existing vector store
vector_store_path = output_dir / "vector_store"

if not vector_store_path.exists():
    print("❌ Error: Existing vector store not found!")
    print("Please run the embedding_generator.ipynb notebook first.")
else:
    print(f"📂 Loading existing vector store from: {vector_store_path}")
    
    # Load the existing FAISS vector store
    vector_store = FAISS.load_local(
        str(vector_store_path),
        embeddings_model,
        allow_dangerous_deserialization=True
    )
    
    print(f"✅ Loaded existing vector store")
    print(f"📊 Current vectors in store: {vector_store.index.ntotal}")


📂 Loading existing vector store from: processed_rules\vector_store
✅ Loaded existing vector store
📊 Current vectors in store: 511


In [18]:
# Convert new chunks to LangChain Documents
def create_langchain_documents(chunks: List[Dict[str, Any]]) -> List[LangChainDocument]:
    """
    Convert our chunks with metadata to LangChain Document objects
    """
    documents = []
    
    for chunk in chunks:
        doc = LangChainDocument(
            page_content=chunk['content'],
            metadata=chunk['metadata']
        )
        documents.append(doc)
    
    return documents

# Create Document objects from our new chunks
print("📄 Converting new chunks to LangChain Documents...")
new_documents = create_langchain_documents(all_new_chunks)

print(f"✅ Created {len(new_documents)} Document objects")

# Preview a sample document
if new_documents:
    sample_doc = new_documents[0]
    print(f"\n🔍 Sample Document:")
    print(f"  Source: {sample_doc.metadata['source']}")
    print(f"  Chunk ID: {sample_doc.metadata['chunk_id']}")
    print(f"  Content preview: {sample_doc.page_content[:200]}...")


📄 Converting new chunks to LangChain Documents...
✅ Created 141 Document objects

🔍 Sample Document:
  Source: twilight_pravila
  Chunk ID: twilight_pravila_chunk_000
  Content preview: General

Dice Values

Q: Are the “0” faces of the ten-sided dice included with the game intended to represent the result of a “10”?

A: Yes, the “0” face is a “10” result.

Commodities

Q: At the star...


In [19]:
# Add new documents to the existing vector store
print("🔥 Adding new documents to vector store...")
print(f"📊 Current vectors: {vector_store.index.ntotal}")
print(f"➕ Adding {len(new_documents)} new vectors")
print("⏳ Generating embeddings and updating vector store...")

try:
    # Add the new documents to the existing vector store
    vector_store.add_documents(new_documents)
    
    print(f"✅ Successfully added new documents!")
    print(f"📈 Total vectors now: {vector_store.index.ntotal}")
    
except Exception as e:
    print(f"❌ Error adding documents: {e}")


🔥 Adding new documents to vector store...
📊 Current vectors: 511
➕ Adding 141 new vectors
⏳ Generating embeddings and updating vector store...
✅ Successfully added new documents!
📈 Total vectors now: 652


In [20]:
# Test the updated vector store with sample queries
print("🧪 Testing updated vector store...")

test_queries = [
    "What are relics?",
    "How do action cards work?",
    "FAQ about combat"
]

print("\n🔍 Sample Query Results:")
print("="*50)

for query in test_queries[:2]:  # Test first 2 queries
    print(f"\n🔸 Query: '{query}'")
    
    try:
        similar_docs = vector_store.similarity_search(
            query=query,
            k=3
        )
        
        print(f"   Found {len(similar_docs)} similar documents:")
        
        for j, doc in enumerate(similar_docs[:2]):
            print(f"   📄 Result {j+1}:")
            print(f"      Source: {doc.metadata['source']}")
            print(f"      Preview: {doc.page_content[:150]}...")
            print()
            
    except Exception as e:
        print(f"   ❌ Error: {e}")

print("✅ Testing complete!")


🧪 Testing updated vector store...

🔍 Sample Query Results:

🔸 Query: 'What are relics?'
   Found 3 similar documents:
   📄 Result 1:
      Source: twilight_pravila
      Preview: Q: What happens to your relics and relic fragments when you're eliminated?

A: Relics are purged, relic fragments discarded.

Q: Can a player “gain” a...

   📄 Result 2:
      Source: rulebook
      Preview: . Then, they place the matching attachment token on that planet on the game board. That planet is modified by the exploration card’s values and abilit...


🔸 Query: 'How do action cards work?'
   Found 3 similar documents:
   📄 Result 1:
      Source: learn_to_play
      Preview: . Action Card Deck Agenda Deck Objective Decks 14 ARCHON TAU 1 1 ARCHON REN 3 2 The was nox dus kill N has fore min STARTING UNITS ✧ ✦1 carrier ✧ ✦2 c...

   📄 Result 2:
      Source: learn_to_play
      Preview: . Many action cards, faction sheets, and even some technology cards have component actions. Each of these effects is pr

In [21]:
# Save the updated vector store
print(f"💾 Saving updated vector store to: {vector_store_path}")

try:
    # Save the updated FAISS vector store
    vector_store.save_local(str(vector_store_path))
    
    print("✅ Updated vector store saved successfully!")
    
    # Update the embedding configuration
    config_path = output_dir / "embedding_config.json"
    
    # Load existing config
    with open(config_path, 'r', encoding='utf-8') as f:
        embedding_config = json.load(f)
    
    # Update with new information
    embedding_config['total_vectors'] = vector_store.index.ntotal
    embedding_config['docx_documents_added'] = len(all_new_chunks)
    embedding_config['sources']['twilight_pravila_chunks'] = len(pravila_chunks)
    embedding_config['sources']['official_faq_chunks'] = len(faq_chunks)
    
    # Save updated config
    with open(config_path, 'w', encoding='utf-8') as f:
        json.dump(embedding_config, f, indent=2, ensure_ascii=False)
    
    print(f"⚙️  Updated configuration saved to: {config_path}")
    
except Exception as e:
    print(f"❌ Error saving vector store: {e}")

print(f"\n🎉 Word Document Processing Complete!")
print(f"📊 Summary:")
print(f"  - Added {len(all_new_chunks)} new chunks from Word documents")
print(f"  - Pravila chunks: {len(pravila_chunks)}")
print(f"  - FAQ chunks: {len(faq_chunks)}")
print(f"  - Total vectors in database: {vector_store.index.ntotal}")
print(f"\n🚀 Your chatbot now has access to all the new content!")


💾 Saving updated vector store to: processed_rules\vector_store
✅ Updated vector store saved successfully!
⚙️  Updated configuration saved to: processed_rules\embedding_config.json

🎉 Word Document Processing Complete!
📊 Summary:
  - Added 141 new chunks from Word documents
  - Pravila chunks: 88
  - FAQ chunks: 53
  - Total vectors in database: 652

🚀 Your chatbot now has access to all the new content!
